In [1]:
"""
MetaData:
---------
Developer: Swapnendu Banik
Description: DQN implementation for CartPole-v1 environment.
Problem Statement: Train an agent to balance a pole on a cart using Deep Q-Network (DQN).
Recommended IDE: Google Colab

Recommended Videos:
-------------------
1. Understanding DQN [Theory]: https://youtu.be/x83WmvbRa2I?si=qUBOc9s--_-TRxOb
2. Understanding SeLU Activation Function Used in Model Arch: https://youtu.be/2OwWs7Hzr9g?si=RlLPiwLWN0pITEVD
3. Code A DQN-Part1 (SentDex) {Diff Arch, Diff Approach, Good Understanding}: https://youtu.be/t3fbETsIBCY?si=YVRa-qdPH2Ayqs-Z
4. Code A DQN-Part2 (SentDex) {Diff Arch, Diff Approach, Good Understanding}: https://youtu.be/qfovbG84EBg?si=zjTuFFVSae5mDIsw


Code Specific:
---------

Operations Carried Out:
------------------------
- Environment setup (OpenAI Gym CartPole-v1)
- Model definition (3-layer fully connected neural network)
- DQN training loop (epsilon-greedy exploration, experience replay)
- Periodic testing and model saving
- Evaluation of trained model performance

Hyperparameters:
----------------
- learning_rate: 1e-4
- gamma: 0.99
- epsilon: 1
- min_epsilon: 0.1
- epsilon_decay: 0.9 / 2.5e3
- memory_len: 10000
- target_update_delay: 2
- test_delay: 10
- episode_limit: 100
- train_batch_size: 100

Environment: CartPole-v1
Algorithm: Deep Q-Network (DQN)
Library: ["PyTorch", "OpenAI-Gym"]
"""

'\nMetaData:\n---------\nDeveloper: Swapnendu Banik\nDescription: DQN implementation for CartPole-v1 environment.\nProblem Statement: Train an agent to balance a pole on a cart using Deep Q-Network (DQN).\nRecommended IDE: Google Colab\n\nRecommended Videos:\n-------------------\n1. Understanding DQN [Theory]: https://youtu.be/x83WmvbRa2I?si=qUBOc9s--_-TRxOb\n2. Understanding SeLU Activation Function Used in Model Arch: https://youtu.be/2OwWs7Hzr9g?si=RlLPiwLWN0pITEVD\n3. Code A DQN-Part1 (SentDex) {Diff Arch, Diff Approach, Good Understanding}: https://youtu.be/t3fbETsIBCY?si=YVRa-qdPH2Ayqs-Z\n4. Code A DQN-Part2 (SentDex) {Diff Arch, Diff Approach, Good Understanding}: https://youtu.be/qfovbG84EBg?si=zjTuFFVSae5mDIsw\n\n\nCode Specific:\n---------\n\nOperations Carried Out:\n------------------------\n- Environment setup (OpenAI Gym CartPole-v1)\n- Model definition (3-layer fully connected neural network)\n- DQN training loop (epsilon-greedy exploration, experience replay)\n- Periodic

In [1]:
from torch import nn
from torch.nn import functional


class Model(nn.Module):
    """
    Defines a fully connected neural network model for Q-learning with three layers.

    Args:
    - input_features (int): Number of input features (state dimension).
    - output_values (int): Number of output values (action space size).
    """
    def __init__(self, input_features, output_values):
        super(Model, self).__init__()  # Initialize the parent class (nn.Module)

        # First fully connected layer (input layer to hidden layer)
        self.fc1 = nn.Linear(in_features=input_features, out_features=32)

        # Second fully connected layer (hidden layer to another hidden layer)
        self.fc2 = nn.Linear(in_features=32, out_features=32)

        # Third fully connected layer (hidden layer to output layer)
        self.fc3 = nn.Linear(in_features=32, out_features=output_values)

    def forward(self, x):
        """
        Defines the forward pass of the neural network. Applies SELU activation
        on the first two layers, and returns the output from the last layer.

        Args:
        - x (tensor): The input tensor (state).

        Returns:
        - tensor: Output Q-values (predicted action values).
        """
        x = functional.selu(self.fc1(x))  # Apply SELU activation to first layer
        x = functional.selu(self.fc2(x))  # Apply SELU activation to second layer
        x = self.fc3(x)  # No activation after the last layer (output layer)
        return x


In [2]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import gym
from collections import deque
import random

In [3]:
import warnings

# Filter the specific warning
warnings.filterwarnings("ignore", category=DeprecationWarning, message="`np.bool8` is a deprecated alias")


c:\Users\nirma\anaconda3\envs\rein_env\lib\site-packages\ipykernel\ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


## Test The Trained Model without rendering Gym Env

In [4]:
use_cuda=True
device = torch.device("cuda" if use_cuda and torch.cuda.is_available() else "cpu")
test_policy_net = Model(4, 2).to(device)  # Adjust input/output dimensions if needed
trained_model_path="trained_weights\\policy_net.pth"
test_policy_net.load_state_dict(torch.load(trained_model_path, map_location=device, weights_only=True))
test_policy_net.eval()  # Set the model to evaluation mode

# Create the environment using new_step_api
env = gym.make('CartPole-v1', new_step_api=True)

def normalize_state(state):
    state[0] /= 2.5
    state[1] /= 2.5
    state[2] /= 0.3
    state[3] /= 0.3

# Number of episodes to evaluate
num_episodes = 10  # You can adjust this as needed

total_rewards = []

for episode in range(num_episodes):
    state, info = env.reset(return_info=True) # get initial state and info
    terminated = False
    truncated = False
    episode_reward = 0

    while not terminated and not truncated:  #check for termination and truncation in loop condition
        # Get action from the loaded policy network
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
        action = test_policy_net(state_tensor).argmax().item()

        # Take action in the environment
        state, reward, terminated, truncated, _ = env.step(action) # get terminated and truncated values
        episode_reward += reward

    print(f"Episode {episode} Reward: {episode_reward}")

    total_rewards.append(episode_reward)

# Calculate average reward
avg_reward = sum(total_rewards) / num_episodes
print(f"Average Reward over {num_episodes} episodes: {avg_reward}")

env.close()  # Close the environment when done

c:\Users\nirma\anaconda3\envs\rein_env\lib\site-packages\ipykernel\ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Episode 0 Reward: 500.0
Episode 1 Reward: 500.0
Episode 2 Reward: 500.0
Episode 3 Reward: 500.0
Episode 4 Reward: 500.0
Episode 5 Reward: 500.0
Episode 6 Reward: 500.0
Episode 7 Reward: 500.0
Episode 8 Reward: 500.0
Episode 9 Reward: 500.0
Average Reward over 10 episodes: 500.0


## Test the model playing the CartPole in real time

In [5]:
def play_cartpole(trained_model_path, use_cuda=True):
    # Set device (use GPU if available, otherwise fallback to CPU)
    device = torch.device("cuda" if use_cuda and torch.cuda.is_available() else "cpu")
    
    # Initialize the model
    test_policy_net = Model(4, 2).to(device)  # Adjust input/output dimensions if needed
    
    # Load the trained model's weights
    test_policy_net.load_state_dict(torch.load(trained_model_path, map_location=device, weights_only=True))
    test_policy_net.eval()  # Set the model to evaluation mode

    # Create the environment
    env = gym.make('CartPole-v1', new_step_api=True,render_mode='human' )

    # Normalize state function (you can modify it as needed)
    def normalize_state(state):
        state[0] /= 2.5
        state[1] /= 2.5
        state[2] /= 0.3
        state[3] /= 0.3

    # Start the environment and reset it
    state, info = env.reset(return_info=True)
    terminated = False
    truncated = False
    episode_reward=0

    # While the game is not over, keep playing
    while not terminated and not truncated:
        # Normalize state (optional step)
        normalize_state(state)

        # Convert state to tensor and send to the device
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
        
        # Get the action from the policy network (model's prediction)
        action = test_policy_net(state_tensor).argmax().item()

        # Take the action in the environment and get new state, reward, done flags
        state, reward, terminated, truncated, _ = env.step(action)
        episode_reward += reward


    print(f"Reward obtained: {episode_reward}")
    # Close the environment when the game is over
    env.close()

In [6]:
play_cartpole(trained_model_path="trained_weights\\policy_net.pth", use_cuda=True)

c:\Users\nirma\anaconda3\envs\rein_env\lib\site-packages\pygame\pkgdata.py:27: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


Reward obtained: 500.0
